# AnyMatch — Inference on AllianceChicago patient pairs (local)

Loads the trained checkpoint and scores candidate patient pairs. Uses synthetic test data here so you can sanity-check the pipeline before plugging in real AllianceChicago data.

**Where to run**: locally, from the `AnyMatch/` folder. Launch Jupyter (or open in VS Code) with cwd = the `AnyMatch/` directory so the relative paths below resolve.

**Hardware**: CPU is fine — GPT-2 inference on 18 pairs takes a couple of seconds. (The underlying `predict()` function only checks for CUDA, so MPS won't be used on Apple Silicon; that's OK at this scale.)

**PHI reminder**: when you swap in real patient data, keep this on a machine that satisfies your institution's policy.

## 1. Paths — where the checkpoint lives

The trained checkpoint sits on Drive at `MyDrive/AnyMatch/checkpoints/anymatch_all9_gpt2/`. To run locally, **download that folder** and place it at:

```
AnyMatch/saved_models/anymatch_all9_gpt2/
```

That directory must contain the HuggingFace files: `model.safetensors` (or `pytorch_model.bin`), `config.json`, `vocab.json`, `merges.txt`, plus optionally `tokenizer_config.json` / `special_tokens_map.json`.

If you placed it somewhere else, edit `CKPT_DIR` in the next cell.

In [1]:
!pip install pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------  9.7/9.9 MB 60.5 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 41.2 MB/s  0:00:00
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ---

In [2]:
import os

# === EDIT THESE IF NEEDED ===
CKPT_DIR   = 'saved_models/anymatch_all9_gpt2'
INPUT_CSV  = 'data/synthetic/alliance_pairs_synthetic.csv'
OUTPUT_CSV = 'data/synthetic/synthetic_predictions.csv'
# ============================

# Sanity check: are we in the right cwd?
assert os.path.exists('loo.py'), (
    f'Expected to run from the AnyMatch/ directory (cwd={os.getcwd()!r}). '
    "Either `cd` into AnyMatch/ before launching Jupyter, or os.chdir() here."
)

# Sanity check: checkpoint files present?
needed = ['config.json', 'vocab.json', 'merges.txt']
missing = [f for f in needed if not os.path.exists(os.path.join(CKPT_DIR, f))]
if missing:
    raise FileNotFoundError(
        f'Missing from {CKPT_DIR}: {missing}. '
        'Download the trained checkpoint folder from Drive '
        '(MyDrive/AnyMatch/checkpoints/anymatch_all9_gpt2/) and place it there.'
    )
print(f'Checkpoint contents at {CKPT_DIR}:')
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(os.path.join(CKPT_DIR, f))
    print(f'  {f:35s} {size/1024:>10.1f} KB')

Checkpoint contents at saved_models/anymatch_all9_gpt2:
  config.json                                0.9 KB
  merges.txt                               445.6 KB
  model.safetensors                     486113.7 KB
  special_tokens_map.json                    0.4 KB
  tokenizer_config.json                      0.5 KB
  vocab.json                               975.8 KB


In [ ]:
# Install deps into the current Python env. Safe to re-run.
# If you're using conda/venv, activate it before launching Jupyter.
import sys
# !pip install -q 'transformers==4.36.2' 'tokenizers<0.20' 'scikit-learn' 'pandas' 'torch'

The system cannot find the file specified.


## 2. Synthetic test data

A small CSV (`data/synthetic/alliance_pairs_synthetic.csv`) ships in the repo with 18 hand-crafted pairs covering the cases real AllianceChicago data should hit:

- **True matches (label=1):** exact, nickname variant (Jonathan/Jon), last-name typo, DOB off by one day, missing middle name on one side, address-format differences, abbreviation differences, missing fields.
- **Non-matches (label=0):** same first name + same DOB but different SSN/address, same name + different DOB, similar-sounding last names, suffix differences (Jr vs Sr).

Schema — every patient attribute appears twice with `_l` / `_r`, plus a `label` column (the `label` is only used for evaluation; real candidate pairs from a blocker won't have it — `predict_alliance.py` adds a placeholder 0 if missing).

In [4]:
import pandas as pd
raw = pd.read_csv(INPUT_CSV)
print(f'{len(raw)} synthetic pairs, label distribution: {raw["label"].value_counts().to_dict()}')
raw.head()

18 synthetic pairs, label distribution: {1: 10, 0: 8}


,first_name_l,middle_name_l,last_name_l,dob_l,ssn4_l,sex_l,address_l,city_l,state_l,zip_l,...,last_name_r,dob_r,ssn4_r,sex_r,address_r,city_r,state_r,zip_r,phone_r,label
0,John,M,Smith,1980-05-12,1234,M,123 Main St,Chicago,IL,60601,...,Smith,1980-05-12,1234.0,M,123 Main St,Chicago,IL,60601,3.125551e+09,1
1,Jonathan,NaN,Smith,1980-05-12,1234,M,123 Main St,Chicago,IL,60601,...,Smith,1980-05-12,1234.0,M,123 Main St,Chicago,IL,60601,3.125551e+09,1
2,Maria,Elena,Gonzalez,1975-09-03,5678,F,456 Oak Ave,Chicago,IL,60614,...,Gonzales,1975-09-03,5678.0,F,456 Oak Ave,Chicago,IL,60614,7.735552e+09,1
3,Robert,J,Johnson,1962-11-21,9012,M,789 Pine Rd,Chicago,IL,60629,...,Johnson,1962-11-22,9012.0,M,789 Pine Rd,Chicago,IL,60629,3.125558e+09,1
4,Latoya,NaN,Williams,1991-02-14,3456,F,2100 W 18th St,Chicago,IL,60608,...,Williams,1991-02-14,NaN,F,2100 W 18th St,Chicago,IL,60608,3.125559e+09,1


## 3. Run inference

Uses the patched `predict_alliance.py` CLI: loads the checkpoint, serializes each row with `df_serializer` (`mode1`), runs the GPT-2 classifier, and writes a CSV with `pred` (0/1) and `match_prob` (P(label=1)) columns appended.

In [10]:
import sys, time, pandas as pd

_n_pairs = len(pd.read_csv(INPUT_CSV))

_t0 = time.perf_counter()
!{sys.executable} predict_alliance.py \
    --model_path {CKPT_DIR} \
    --base_model gpt2 \
    --serialization_mode mode1 \
    --input_csv {INPUT_CSV} \
    --output_csv {OUTPUT_CSV} \
    --batch_size 32
_t1 = time.perf_counter()

_total_s = _t1 - _t0
print(f'\nTotal inference time : {_total_s:.2f} s')
print(f'Pairs scored         : {_n_pairs}')
print(f'Time per pair        : {_total_s / _n_pairs * 1000:.1f} ms')

0 out of 18 data samples are filtered.
Wrote 18 predictions to data/synthetic/synthetic_predictions.csv

Total inference time : 12.60 s
Pairs scored         : 18
Time per pair        : 699.8 ms


c:\Users\MiguelGarcia\.conda\envs\llm_env\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\MiguelGarcia\.conda\envs\llm_env\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


## 4. Inspect predictions

On the synthetic set we have ground truth, so we can compute accuracy / F1 / per-row results.

In [11]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report

out = pd.read_csv(OUTPUT_CSV)

view_cols = ['first_name_l', 'last_name_l', 'dob_l',
             'first_name_r', 'last_name_r', 'dob_r',
             'label', 'pred', 'match_prob']
print('--- Per-row predictions ---')
display(out[view_cols].round({'match_prob': 3}))

if 'label' in out.columns and out['label'].notna().any():
    print('\n--- Metrics vs. ground truth ---')
    f1  = f1_score(out['label'], out['pred'])
    acc = accuracy_score(out['label'], out['pred'])
    print(f'accuracy = {acc*100:.1f}%   F1 = {f1*100:.1f}%')
    print('\n', classification_report(out['label'], out['pred'], digits=3))

--- Per-row predictions ---


,first_name_l,last_name_l,dob_l,first_name_r,last_name_r,dob_r,label,pred,match_prob
0,John,Smith,1980-05-12,John,Smith,1980-05-12,1,1,0.999
1,Jonathan,Smith,1980-05-12,Jon,Smith,1980-05-12,1,1,0.999
2,Maria,Gonzalez,1975-09-03,Maria,Gonzales,1975-09-03,1,1,0.999
3,Robert,Johnson,1962-11-21,Robert,Johnson,1962-11-22,1,1,0.939
4,Latoya,Williams,1991-02-14,LaToya,Williams,1991-02-14,1,1,0.957
5,Michael,Brown,1955-07-30,Mike,Brown,1955-07-30,1,1,0.998
6,Sarah,Lee,1988-12-01,Sarah,Lee,1988-12-01,1,1,0.980
7,Carlos,Rivera,1970-04-17,Carlos,Rivera,1970-04-17,1,1,0.997
8,Aisha,Patel,1995-08-25,Aisha,Patel,1995-08-25,1,1,0.997
9,James,O'Brien,1948-03-09,James,OBrien,1948-03-09,1,1,0.999



--- Metrics vs. ground truth ---
accuracy = 94.4%   F1 = 95.2%

               precision    recall  f1-score   support

           0      1.000     0.875     0.933         8
           1      0.909     1.000     0.952        10

    accuracy                          0.944        18
   macro avg      0.955     0.938     0.943        18
weighted avg      0.949     0.944     0.944        18



## 5. Plug in real AllianceChicago pairs

When you have real candidate pairs:

1. Save them as a CSV with the **same `_l` / `_r` schema** the synthetic file uses (column names can differ; only the `_l` / `_r` suffixing matters).
2. Place the CSV anywhere (e.g. `data/alliance/real_pairs.csv`).
3. Re-point `INPUT_CSV` and `OUTPUT_CSV` in cell 2, then re-run Section 3.

Notes:
- Normalize case, whitespace, and date format upstream — AnyMatch is a string-comparison model and sensitive to formatting drift.
- For tuning, use `match_prob` (not just `pred`) so you can sweep a threshold for your precision / recall target.
- If pairs are unlabeled, simply omit the `label` column — `predict_alliance.py` will add a placeholder.
- For batches larger than a few thousand pairs on CPU, expect a few minutes per 1k pairs. If that's too slow, drop the file on Colab and use the original `anymatch_colab.ipynb` Section 5 instead.